# SI10-2026 | Ponderada | Análise de Sensibilidade em Métricas de Interface Digital

Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.precision", 3)

## Dados

Execute a célula abaixo para criar a base da atividade.

In [2]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [3]:
# Use esta célula para criar sua análise exploratória.

colunas_numericas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
    "taxa_conversao_pct",
]

df[colunas_numericas].corr()

,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
taxa_abandono_carrinho_pct,1.000,-0.068,-0.116,-0.643
profundidade_scroll_pct,-0.068,1.000,0.050,0.485
tempo_primeiro_clique_s,-0.116,0.050,1.000,-0.229
taxa_conversao_pct,-0.643,0.485,-0.229,1.000


In [8]:
# Preencha com uma variável de entrada para visualizar.
# Use exatamente um dos nomes que aparecem em features.

features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]

variavel_x = ["taxa_abandono_carrinho_pct", "profundidade_scroll_pct"]
for variavel in variavel_x:
  if variavel not in features:
      raise ValueError("Preencha variavel_x com uma variável da lista features.")

  fig = px.scatter(
      df,
      x=variavel,
      y="taxa_conversao_pct",
      title="Relação com a taxa de conversão",
  )
  fig.show()

Escreva quais duas variáveis você escolheu para a análise de sensibilidade e justifique com evidências da exploração.

**Resposta:**

Escolhi taxa_abandono_carrinho_pct e profundidade_scroll_pct.

A matriz de correlação mostra que o abandono do carrinho tem a correlação mais forte com a taxa de conversão (r = -0,64) e a profundidade de scroll vem em segundo (r = +0,48). O tempo até o primeiro clique ficou bem abaixo dos dois (r = -0,23), então não faz sentido priorizá-lo.

O scatter do abandono contra a conversão confirma a tendência negativa: dias com abandono alto tendem a ter conversão baixa. Com base nisso, essas duas são as variáveis com maior potencial de impacto e foram escolhidas para a análise de sensibilidade.

## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [9]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))

pd.DataFrame({
    "métrica": ["MAE", "RMSE"],
    "valor": [mae, rmse],
})

,métrica,valor
0,MAE,0.276
1,RMSE,0.344


Interprete o erro do modelo em relação à taxa de conversão.

**Resposta:**

O modelo apresenta MAE de 0,28 pontos percentuais e RMSE de 0,34. A taxa de conversão nos dados varia entre 4,4% e 7,8%, com média de 5,87%, então o erro médio representa menos de 5% do valor típico, o que é razoável para dados de comportamento de usuário que têm bastante variação natural no dia a dia.

O RMSE um pouco acima do MAE indica que em alguns dias o modelo errou mais do que a média, mas sem nenhum outlier grave que distorcesse os resultados. Na prática, esse nível de erro não compromete a análise de sensibilidade porque o que importa é a direção e a proporção do impacto de cada variável, não a previsão exata de cada dia.

## Parte 3: Análise de Sensibilidade

Calcule a sensibilidade para duas variáveis de entrada usando uma variação de 10%.

Use a fórmula: sensibilidade igual à variação percentual da saída dividida pela variação percentual da entrada.

In [10]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

linha_base, saida_base

({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032,
  'tempo_primeiro_clique_s': 6.9708957453209095},
 5.868747841831934)

In [11]:
# Preencha com duas variáveis escolhidas na Parte 1.
# Use exatamente os nomes que aparecem em features.

variaveis_escolhidas = ["taxa_abandono_carrinho_pct", "profundidade_scroll_pct"]

if len(variaveis_escolhidas) != 2:
    raise ValueError("Preencha variaveis_escolhidas com duas variáveis da lista features.")

variaveis_invalidas = [v for v in variaveis_escolhidas if v not in features]

if variaveis_invalidas:
    raise ValueError(f"Variáveis fora de features: {variaveis_invalidas}")

variacao_entrada = 0.10

resultados = []

for variavel in variaveis_escolhidas:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada

    resultados.append({
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saida_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
    })

tabela_sensibilidade = pd.DataFrame(resultados)
tabela_sensibilidade

,variável,valor_original,valor_alterado,saída_original,saída_nova,variação_saida_pct,índice_sensibilidade
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.582,-4.891,-0.489
1,profundidade_scroll_pct,62.449,68.694,5.869,6.020,2.578,0.258


Compare os índices de sensibilidade e indique qual variável tem maior impacto sobre a taxa de conversão.

Mostre o raciocínio: cite os valores da tabela e explique o que eles significam para a decisão.

**Resposta:**

O abandono do carrinho tem índice de sensibilidade de -0,49: um aumento de 10% nessa variável reduz a taxa de conversão em 4,89%. A profundidade de scroll tem índice de +0,26: um aumento de 10% melhora a conversão em 2,58%. Em valor absoluto, o abandono causa quase o dobro do impacto que o scroll, então ele é a variável de maior peso na decisão.

Vale notar que o tempo até o primeiro clique tem o maior coeficiente bruto no modelo (-0,094), mas seu índice de sensibilidade seria o mais baixo dos três (-0,11). Isso acontece porque a média dele é pequena (~7s), então 10% de variação equivale a menos de 1 segundo não é o suficiente para mover a conversão de forma relevante. A sensibilidade captura esse efeito que olhar só para o coeficiente não mostraria.

## Parte 4: Decisão

Recomende uma ação de produto ou interface com base na análise.

Sua recomendação deve citar os números da tabela de sensibilidade.

**Resposta:**

A prioridade deve ser reduzir a taxa de abandono do carrinho. Com índice de sensibilidade de -0,49, uma queda de 10% nessa variável, saindo de 47,5% para cerca de 42,8%, geraria uma melhora de 4,89% na taxa de conversão. É o maior retorno possível entre as variáveis analisadas, quase o dobro do que conseguiríamos mexendo no scroll.

Na prática, isso significa olhar para o fluxo de checkout e entender onde os usuários estão desistindo. Pode ser excesso de etapas, formulários longos, falta de opções de pagamento ou simplesmente o carrinho não salvar quando o usuário sai e volta. Qualquer uma dessas mudanças tem potencial de mover o abandono na direção certa.

O scroll tem impacto real também (índice +0,26), mas é uma alavanca mais difícil de controlar porque depende do usuário já estar engajado o suficiente para rolar a página. Faz mais sentido como melhoria secundária, depois de resolver o problema do abandono.

Aponte uma limitação, risco ou hipótese da sua análise.

**Resposta:**

O modelo assume que a relação entre as variáveis e a conversão é linear, o que dificilmente reflete o comportamento real de usuários. Na prática existem pontos de saturação: reduzir o abandono de 47% para 42% pode gerar um ganho, mas reduzir de 20% para 15% provavelmente não gera o mesmo efeito proporcional porque quem ainda abandona nesse nível tem motivos bem diferentes.

Além disso, a análise de sensibilidade perturba uma variável por vez mantendo as outras completamente fixas. Isso raramente acontece numa mudança real de produto: simplificar o checkout provavelmente mexe no abandono, mas pode também alterar o tempo de clique e até o padrão de scroll dos usuários. Os índices calculados são válidos como referência para priorização, mas não devem ser lidos como previsão de resultado de uma mudança real.

## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

In [12]:
# Use esta célula para sua simulação.

n_simulacoes = 1000

amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng.normal(
        linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes
    ).clip(25, 75),
    "profundidade_scroll_pct": rng.normal(
        linha_base["profundidade_scroll_pct"], 8, n_simulacoes
    ).clip(25, 95),
    "tempo_primeiro_clique_s": rng.normal(
        linha_base["tempo_primeiro_clique_s"], 1.5, n_simulacoes
    ).clip(2, 15),
})

amostras_design = np.column_stack([
    np.ones(len(amostras)),
    amostras[features].to_numpy(),
])
previsoes = amostras_design @ coeficientes

pd.Series(previsoes).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

,0
count,1000.000
mean,5.869
std,0.393
min,4.654
10%,5.369
25%,5.617
50%,5.854
75%,6.131
90%,6.379
max,7.193


In [13]:
fig = px.histogram(
    pd.DataFrame({"taxa_conversao_pct_prevista": previsoes}),
    x="taxa_conversao_pct_prevista",
    nbins=30,
    title="Distribuição simulada da taxa de conversão",
)
fig.show()

Interprete o que a distribuição simulada indica sobre o risco da sua recomendação.

**Resposta:**

A simulação rodou 1000 cenários jogando variação aleatória nas três entradas ao redor dos valores médios. A conversão média ficou em 5,87% com desvio de 0,39, o que é bem comportado dado que o abandono foi perturbado em até 5 pontos, o scroll em 8 e o tempo de clique em 1,5 segundo.

Entre o P10 e o P90 a conversão ficou de 5,37% a 6,38%, então em 80% dos casos simulados ela variou menos de 1 ponto percentual. O pior cenário das 1000 rodadas foi 4,65% e o melhor chegou a 7,19%, próximo do máximo que aparece nos dados reais.

O que a simulação mostra é que o modelo é estável, mas ela não responde se uma mudança de interface consegue de fato baixar o abandono em 10% sem mexer nas outras métricas junto. Esse é o risco real da recomendação, e só um teste com usuários reais resolveria.

## Política de Uso de IA

O uso de IA é permitido para apoio técnico, revisão de texto e estudo dos conceitos.

As escolhas de variáveis, os cálculos, a comparação dos índices e a recomendação devem refletir sua análise dos resultados deste notebook.

Você deve ser capaz de explicar qualquer resposta entregue.

Respostas sem relação com os números gerados, com indícios de cópia ou que não possam ser justificadas poderão ser tratadas como fora da proposta.

## Instruções de entrega

A entrega deverá ser feita no GitHub ou no próprio Google Colab.

Links **sem permissão** de acesso terão um desconto de 20% na nota.